# Recyclens: the second opinion, served from Colab

`prithivMLmods/Recycling-Net-11` behind a public HTTPS URL, so the vision proxy
has somewhere to send a photo.

**Why this notebook exists.** Two doors were shut first. Hugging Face serves no
inference provider for this model (`inferenceProviderMapping` is `{}`), so there
is no serverless URL to call and an access token does not change that. And
creating a Space that runs on compute now requires a paid HF plan, free CPU
hardware or not.

So the model runs here, on Colab's free CPU, and a Cloudflare quick tunnel gives
it a public URL. No account, no card, no HF plan.

**What it costs you.** The URL changes every session and Colab drops the runtime
after roughly 90 minutes idle. Re-run the notebook and paste the new URL into
Vercel. That is the trade for free. If you want a URL that never moves, HF PRO is
$9/month and `classifier/README.md` deploys the same `app.py` as a Space instead.

**It runs the real `app.py`**, uploaded from the repo rather than retyped here, so
what this serves is exactly what a Space would serve: the same weights, the same
`SiglipImageProcessor`, the same EXIF transpose and square centre crop.


## 1. Install


In [ ]:
!pip install -q transformers==4.50.3 pillow==11.1.0 fastapi==0.115.8 'uvicorn[standard]==0.34.0'

# torch already ships on Colab, so its version is whatever the runtime has.
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)


## 2. Upload `classifier/app.py`

From your repo. Uploading it rather than pasting its contents here is the point:
there is then only one copy of the preprocessing, and this notebook cannot drift
away from the thing you would otherwise deploy.


In [ ]:
from google.colab import files

uploaded = files.upload()  # pick classifier/app.py
assert 'app.py' in uploaded, 'upload classifier/app.py, not something else'
print('ok')


## 3. Choose the secret and the crop

`CLASSIFIER_TOKEN` is any long random string. The proxy sends it as a bearer
token and `app.py` refuses anything else. A quick tunnel URL is guessable, so set
one rather than leaving the endpoint open.

`CLASSIFIER_CROP` is the judgement call worth measuring. `square` centre crops
before the processor's 224x224 resize, so the item changes scale and never shape.
`none` passes the whole frame and lets the processor squash it, which is what the
model's own training script did.


In [ ]:
import os, secrets

CLASSIFIER_TOKEN = secrets.token_urlsafe(32)
CLASSIFIER_CROP = 'square'   # or 'none', to compare

# Set BEFORE importing app, which reads both at import time.
os.environ['CLASSIFIER_TOKEN'] = CLASSIFIER_TOKEN
os.environ['CLASSIFIER_CROP'] = CLASSIFIER_CROP
print('token:', CLASSIFIER_TOKEN)


## 4. Load the model

The first run downloads 372MB of weights, about a minute.


In [ ]:
import time

started = time.time()
import app
print(f'loaded in {time.time() - started:.0f}s')
print('model  :', app.MODEL_ID)
print('crop   :', app.CROP)
print('labels :', ', '.join(app.LABELS))
print('process:', app.processor.size, 'mean', app.processor.image_mean,
      'std', app.processor.image_std)


## 5. Serve it, and open a tunnel


In [ ]:
import socket, threading, uvicorn

config = uvicorn.Config(app.app, host='0.0.0.0', port=7860, log_level='warning')
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()

# Wait for the port to bind rather than sleeping a guessed number of seconds.
while True:
    try:
        socket.create_connection(('127.0.0.1', 7860), timeout=1).close()
        break
    except OSError:
        time.sleep(0.5)

print('serving on :7860')


In [ ]:
# A Cloudflare quick tunnel: a public HTTPS URL, no account and no card.
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared

import re, subprocess

proc = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:7860', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

PUBLIC_URL = None
for line in proc.stdout:
    found = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if found:
        PUBLIC_URL = found.group(0)
        break

assert PUBLIC_URL, 'no tunnel URL appeared, re-run this cell'
print(PUBLIC_URL)


## 6. Check it, then wire the proxy


In [ ]:
import requests

health = requests.get(f'{PUBLIC_URL}/health', timeout=60).json()
print('model    :', health['model'])
print('crop     :', health['crop'])
print('protected:', health['protected'])
print()
print('Set these two in Vercel (the proxy project, Settings, Environment')
print('Variables), then redeploy the proxy:')
print()
print(f'  CLASSIFIER_URL={PUBLIC_URL}/classify')
print(f'  CLASSIFIER_TOKEN={CLASSIFIER_TOKEN}')


## 7. Classify a photo

Upload one of your packaging photos. This is the same call the proxy makes.


In [ ]:
shots = files.upload()

for name, raw in shots.items():
    reply = requests.post(
        f'{PUBLIC_URL}/classify',
        data=raw,
        headers={
            'content-type': 'image/jpeg',
            'authorization': f'Bearer {CLASSIFIER_TOKEN}',
        },
        timeout=120,
    )
    rows = reply.json()
    print(name)
    for row in rows[:4]:
        print(f"    {row['label']:<18} {row['score'] * 100:5.1f}%")
    print()


## 8. The crop experiment, for the report

The one cheap measurable result in this feature. Run your whole photo set through
both settings and count how often the top label changes, and in which direction.
To switch, set `CLASSIFIER_CROP = 'none'` in step 3, restart the runtime, and run
again.

Nothing here sets a pass bar. The classifier was trained on a different dataset
and is expected to be wrong sometimes, which is the whole reason the rule engine
only ever lets it refuse an answer and never give one.


In [ ]:
photos = files.upload()

results = {}
for name, raw in photos.items():
    rows = requests.post(
        f'{PUBLIC_URL}/classify',
        data=raw,
        headers={
            'content-type': 'image/jpeg',
            'authorization': f'Bearer {CLASSIFIER_TOKEN}',
        },
        timeout=120,
    ).json()
    results[name] = rows[0]
    print(f"{name:<30} {rows[0]['label']:<18} {rows[0]['score'] * 100:5.1f}%")

print()
print(f'crop={app.CROP}, {len(results)} photos')
